# Homework 1: Agentic RAG

[Homework instructions](https://github.com/DataTalksClub/llm-zoomcamp/blob/af60c5a0ad93c75c7fb7617330825022e3ef489d/cohorts/2026/homework/01-agentic-rag/homework.md)

## Setup

Load `.env`, create the OpenAI client.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

## Data

Read `DataTalksClub/llm-zoomcamp` at commit `8c1834d` with `GithubRepositoryDataReader` (`md` files, paths containing `/lessons/`), then `parse()` each file into `filename` + `content`.

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q1. Number of lesson pages

In [3]:
len(documents)

72

## Q2. minsearch index

`content` as text field, `filename` as keyword field.

In [4]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

[r['filename'] for r in search_results]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/11-agents-intro.md',
 '01-agentic-rag/lessons/16-other-frameworks.md']

## Q3. RAG input tokens

RAG flow over the Q2 index: `search` → build context from `filename` + `content` → call the LLM, keeping the full response so `usage.input_tokens` is available. Model: `gpt-5.4-mini`.

In [6]:
instructions = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

prompt_template = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

context = []

for doc in search_results:
    context.append('filename: ' + doc['filename'])
    context.append('content: ' + doc['content'])
    context.append('')

context = '\n'.join(context).strip()

len(context)

29852

In [7]:

question = "How does the agentic loop keep calling the model until it stops?"
prompt = prompt_template.format(
    question=question, context=context
)

print(prompt[:500])

QUESTION: How does the agentic loop keep calling the model until it stops?

CONTEXT:
filename: 01-agentic-rag/lessons/14-agentic-loop.md
content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the mode


In [8]:
messages = [
  {'role': 'developer', 'content': instructions},
  {'role': 'user', 'content': prompt}
]

In [9]:
model="gpt-5.4-mini"
response = openai_client.responses.create(
  model=model,
  input=messages
)

answer = response.output_text

answer

'It keeps calling the model inside a `while True` loop, then checks whether the response contained any `function_call` items.\n\n- If there is a function call, it runs the tool, appends the tool result to `messages`, and loops again.\n- If there are no function calls, it breaks out of the loop.\n\nSo the exit condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn short: it repeats until the model returns a final message with no tool calls.'

In [10]:
response.usage.input_tokens

7136

## Q4. Chunking

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [12]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

## Q5. RAG on chunks

Re-run the Q3 query on the chunk index and compare input tokens.

In [13]:
from minsearch import Index

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [14]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = chunk_index.search(
    question,
    num_results=5
)

search_results[0]


{'start': 4000,
 'content': 'while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\niteration counter so we can see how many round-trips happened.\n\n```python\nit = 1\n\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n\n        elif item.type == "message":\n            print("ASSISTANT:")\n            print(item.content[0].text)\n\n    it = it + 1\n    if has_function_calls == False:\n        break\n```\n\nThis is the core agent loop. The model reaso

In [15]:

instructions = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

prompt_template = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

context = []

for chunk in search_results:
    context.append('filename: ' + chunk['filename'])
    context.append('content: ' + chunk['content'])
    context.append('')

context = '\n'.join(context).strip()

# me: we only add the search results to the context,
len(context)

9688

In [16]:
question = "How does the agentic loop keep calling the model until it stops?"
prompt = prompt_template.format(
    question=question, context=context
)

print(prompt[:500])

QUESTION: How does the agentic loop keep calling the model until it stops?

CONTEXT:
filename: 01-agentic-rag/lessons/14-agentic-loop.md
content: while` loop. The loop keeps calling the model until
it returns a response without any function calls. We also keep an
iteration counter so we can see how many round-trips happened.

```python
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",



In [17]:
messages = [
  {'role': 'developer', 'content': instructions},
  {'role': 'user', 'content': prompt}
]

In [18]:
model="gpt-5.4-mini"
response = openai_client.responses.create(
  model=model,
  input=messages
)

answer = response.output_text

answer

'The loop keeps calling the model inside a `while True` loop, and after each call it checks whether any `function_call` items were returned.\n\n- If there is at least one function call, it runs the tool, appends the tool result to `messages`, and loops again.\n- If there are no function calls in that turn, it breaks out of the loop.\n\nSo the stop condition is: **no function calls this turn**.'

In [19]:
response.usage.input_tokens

2319

## Q6. Agent with a `search` tool

Count how many times `search` is called.

In [20]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the content database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course content."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [21]:
def search(query: str) -> list[dict]:
    """Search the course lessons for chunks matching the query."""
    return chunk_index.search(
        query,
        num_results=5,
    )

In [22]:
import json

def make_call(call):
    args = json.loads(call.arguments)
    print(args)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [23]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

question = "How does the agentic loop work, and how is it different from plain RAG?"

messages = [
  {'role': 'developer', 'content': instructions},
  {'role': 'user', 'content': question}
]

messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant. \nAnswer the student's question using the search tool. \nMake multiple searches with different keywords before answering.\n"},
 {'role': 'user',
  'content': 'How does the agentic loop work, and how is it different from plain RAG?'}]

In [24]:
iteration = 1
tool_count = 0

while True:
    print(f"iteration #{iteration}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            tool_count = tool_count + 1
            print("toolCount:", tool_count)
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    iteration = iteration + 1
    if has_function_calls == False:
        break

iteration #1...
toolCount: 1
function_call: search {"query":"agentic loop RAG difference"}
{'query': 'agentic loop RAG difference'}
toolCount: 2
function_call: search {"query":"agentic loop retrieve reason act"}
{'query': 'agentic loop retrieve reason act'}
toolCount: 3
function_call: search {"query":"plain RAG vs agentic loop"}
{'query': 'plain RAG vs agentic loop'}
iteration #2...
ASSISTANT:
The agentic loop is a repeated **LLM → tool call → tool result → LLM** cycle.

From the course material, it works like this:

1. Send the user prompt to the model.
2. The model may return a tool/function call instead of a final answer.
3. Your code executes that tool.
4. You send the tool result back to the model.
5. Repeat until the model returns a final answer with no more tool calls.

The lesson describes it as a `while` loop that keeps going until the model is done.

### How it differs from plain RAG

Plain RAG is basically a fixed pipeline:

```python
search_results = search(question)
user_p